# exp_air_gis

## 0. 开头与公共准备

### 0.1 Notebook 目标与结构说明

本实验在 `air` 数据基础上，按照数据驱动 `GIS` 主流程重新组织分析：先从空气质量时间序列中构造微观观测，再拟合微观层线性 `GIS`，随后用粗粒化矩阵 `W` 生成宏观变量，并比较宏微两层的动力学效率。

主线中会反复用到下面几组量：

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}\mathbf{o}_t + \boldsymbol{\varepsilon}_t,
\qquad
\boldsymbol{\varepsilon}_t \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma}),
$$

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}^{\top}\boldsymbol{\Sigma}^{-1}\mathbf{A}),
$$

$$
\log \Gamma_{\alpha}^{\mathrm{GIS}} = \left(\frac12 - \frac{\alpha}{4}\right)N + \frac{\alpha}{4}D,
\qquad
J_{\alpha} = \frac{1}{d}\log \Gamma_{\alpha}^{\mathrm{GIS}},
$$

以及宏观效率增益：

$$
\mathrm{CE} = J_{\alpha}^{\text{macro}} - J_{\alpha}^{\text{micro}}.
$$

下面的 notebook 按“解释块 + 代码块”交替展开，便于直接重跑，也方便之后改筛选区域、改变量组合、改宏观维度。


In [ ]:
notebook_outline = {
    'part_0': ['0.1 目标与结构', '0.2 公共依赖与统一参数', '0.3 本实验特有函数'],
    'part_1': ['1.1 数据读取', '1.2 观测函数与样本配对', '1.3 微观层拟合', '1.4 微观层预测', '1.5 微观层 GIS',
               '1.6 宏观维度与 W', '1.7 宏观数据', '1.8 宏观层拟合', '1.9 宏观层预测', '1.10 宏观层 GIS',
               '1.11 CE 与曲线对比', '1.12 结果汇总'],
    'part_2': ['2.1 粗粒化矩阵地图归因', '2.2 频谱分析'],
    'part_3': ['3.1 统一结论'],
}
notebook_outline

### 0.2 公共依赖、绘图风格与统一参数

这一部分完成环境准备、绘图风格统一和实验参数集中配置。这里仍沿用 `exp_air_0321.ipynb` 的数据读取方式，但主流程改为数据驱动 `GIS` 路线，因此参数重点放在：

1. 观测函数的时间延迟嵌入；
2. 配对时间尺度 `tau`；
3. 微观到宏观的维度选择方式；
4. 预测误差和 `GIS` 指标的统一对比；
5. 数据区域筛选和变量组合的自由切换。

为了后续切换方便，筛选参数都集中放在 `config['subset']` 和 `config['variables']` 中：

- 若想用全部站点，可设 `subset_mode='all'`；
- 若想只看单个省，可设 `subset_mode='province'` 且给出一个 `province_names`；
- 若想看多个省，可设 `subset_mode='provinces'`；
- 若想看单个市或多个市，可设 `subset_mode='city'` / `subset_mode='cities'`；
- 若想自定义混合筛选，可设 `subset_mode='custom'`，并结合省、市、站点列表；
- 变量默认是 `['PM2.5', 'O3']`，但也可以替换成数据集中可用的任意变量组合。


In [ ]:
import sys
import types
import re
from pathlib import Path

REPO_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / 'tools.py').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing tools.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
if not hasattr(np, 'typeDict'):
    np.typeDict = np.sctypeDict

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError:
    plotly_module = types.ModuleType('plotly')
    plotly_express = types.ModuleType('plotly.express')
    plotly_graph_objects = types.ModuleType('plotly.graph_objects')
    plotly_module.express = plotly_express
    plotly_module.graph_objects = plotly_graph_objects
    sys.modules['plotly'] = plotly_module
    sys.modules['plotly.express'] = plotly_express
    sys.modules['plotly.graph_objects'] = plotly_graph_objects

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from tools import (
    load_air_data_subset,
    build_air_feature_matrix,
    summarize_air_subset,
    lift_time_delay,
    prepare_time_pairs,
    fit_linear_gis_from_pairs,
    compute_gis_metrics,
    compute_prediction_errors,
    compute_ce_from_micro_macro,
    select_macro_rank,
    build_w_from_svd,
    apply_coarse_graining,
    summarize_pipeline_results,
    plot_station_with_map,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei',
    'SimHei',
    'Noto Sans CJK SC',
    'Source Han Sans SC',
    'Arial Unicode MS',
    'DejaVu Sans',
]
HEATMAP_CMAP = 'vlag'

config = {
    'experiment_name': 'exp_air_gis',
    'dataset_path_candidates': [
        REPO_ROOT / 'data' / 'dataset_yrd.nc',
        REPO_ROOT / 'data' / 'dataset_yrd.nc',
    ],
    'station_meta_path_candidates': [
        REPO_ROOT / 'data' / 'stations_yrd.csv',
        REPO_ROOT / 'data' / 'stations_yrd.csv',
    ],
    'subset': {
        'subset_mode': 'all',
        'province_names': None,
        'city_names': None,
        'station_ids': None,
        'time_slice': None,
        # 示例 1：单省
        # 'subset_mode': 'province',
        # 'province_names': ['江苏省'],
        # 示例 2：多个省
        # 'subset_mode': 'provinces',
        # 'province_names': ['江苏省', '浙江省'],
        # 示例 3：多个市
        # 'subset_mode': 'cities',
        # 'city_names': ['上海市', '南京市', '苏州市'],
        # 示例 4：自定义混合筛选
        # 'subset_mode': 'custom',
        # 'province_names': ['江苏省'],
        # 'city_names': ['上海市'],
        # 'station_ids': ['1141A', '1142A'],
    },
    'variables': ['PM2.5', 'O3'],
    # 也可以改成 ['PM2.5']、['O3']、['PM10', 'NO2'] 等数据集中存在的组合
    'n_delays': 2,
    'delay_interval': 24,
    'tau': 24,
    'burn_in': 0,
    'stride': 1,
    'alpha': 1.0,
    'eps': 1e-10,
    'ridge': 1e-8,
    'rank_mode': 'manual',
    'manual_r': 2,
    'rank_threshold': None,
    'horizons': (1, 3, 5),
    'selected_station_plot_count': 2,
    'max_micro_plot_count': 6,
    'matrix_max_visible_labels': 18,
    'map_per_panel_size': (4.2, 5.0),
}

config

### 0.3 公共函数区（本实验特有的，不包括复用 `tools.py` 里面的函数）

这里把本实验自己补充的辅助函数放在一起，尽量不改动仓库已有实现。主要补三类能力：

1. 适配大矩阵展示的稀疏标签与热力图；
2. 空气数据变量名、延迟标签和站点地图面板的整理；
3. 曲线对比与频谱分析时需要的小工具。

如果后面发现已有函数不够用，优先继续往这一块补，而不是去改其他文件。


In [ ]:
def locate_existing_path(candidates, label):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    raise FileNotFoundError(f'Could not locate {label}. Tried: {candidates}')


def normalize_air_variable_name(name):
    alias = {'PM2.5': 'pm25', 'O3': 'o3'}
    name = str(name)
    if name in alias:
        return alias[name]
    return name.lower().replace('.', '').replace(' ', '_')


def simplify_feature_name(name):
    name = str(name)
    name = name.replace('PM2.5', 'pm25').replace('O3', 'o3')
    return name


def sparse_labels(labels, max_visible=18):
    if labels is None:
        return False
    labels = list(labels)
    if len(labels) <= max_visible:
        return labels
    step = max(1, int(np.ceil(len(labels) / max_visible)))
    return [label if idx % step == 0 else '' for idx, label in enumerate(labels)]


def plot_matrix_heatmap(matrix, title, row_labels=None, col_labels=None, center=0.0, figsize=(6, 6), cmap=HEATMAP_CMAP, max_visible_labels=18):
    matrix = np.asarray(matrix, dtype=float)
    n_rows, n_cols = matrix.shape
    is_square = n_rows == n_cols
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        matrix,
        ax=ax,
        cmap=cmap,
        center=center,
        square=is_square,
        xticklabels=sparse_labels(col_labels, max_visible=max_visible_labels),
        yticklabels=sparse_labels(row_labels, max_visible=max_visible_labels),
        cbar_kws={'shrink': 0.72, 'fraction': 0.035, 'pad': 0.02},
    )
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90, labelsize=6.5)
    ax.tick_params(axis='y', rotation=0, labelsize=6.5)
    plt.tight_layout()
    plt.show()


def plot_dual_singular_spectra(sv_forward, sv_backward, title, max_points=None, figsize=(8.6, 4.8)):
    sv_forward = np.asarray(sv_forward, dtype=float).ravel()
    sv_backward = np.asarray(sv_backward, dtype=float).ravel()
    n = max(len(sv_forward), len(sv_backward))
    if max_points is not None:
        n = min(n, int(max_points))
    x = np.arange(1, n + 1)
    fwd = sv_forward[:n]
    bwd = sv_backward[:n]
    width = 0.42
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(x - width / 2, fwd, width=width, color='tab:blue', alpha=0.35, label='前向谱 $\\Sigma^{-1}$')
    ax.plot(x - width / 2, fwd, color='tab:blue', marker='o', linewidth=1.6)
    ax.bar(x + width / 2, bwd, width=width, color='tab:orange', alpha=0.35, label='后向谱 $A^T\\Sigma^{-1}A$')
    ax.plot(x + width / 2, bwd, color='tab:orange', marker='s', linewidth=1.6)
    ax.set_title(title)
    ax.set_xlabel('奇异值序号')
    ax.set_ylabel('奇异值')
    ax.legend()
    plt.tight_layout()
    plt.show()


def standardize_for_plot(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)


def infer_sampling_hours(times):
    times = pd.to_datetime(np.asarray(times))
    if len(times) < 2:
        return 1.0
    diffs = np.diff(times.values.astype('datetime64[m]').astype('int64')) / 60.0
    diffs = diffs[np.isfinite(diffs)]
    return float(np.median(diffs)) if len(diffs) else 1.0


def extract_feature_meta(feature_names):
    rows = []
    pattern = re.compile(r'^(?P<station>[^_]+)_(?P<var>[^_]+?)(?:_d(?P<delay>\\d+))?$')
    for idx, raw_name in enumerate(feature_names):
        name = simplify_feature_name(raw_name)
        matched = pattern.match(name)
        if matched is None:
            rows.append({
                'feature_index': idx,
                'feature_name': name,
                'station_id': name,
                'variable': 'unknown',
                'delay': 0,
            })
            continue
        rows.append({
            'feature_index': idx,
            'feature_name': name,
            'station_id': matched.group('station'),
            'variable': normalize_air_variable_name(matched.group('var')),
            'delay': int(matched.group('delay') or 0),
        })
    return pd.DataFrame(rows)


def build_selected_micro_indices(feature_meta, variables, n_station_each=2, max_total=6):
    picked = []
    for var in variables:
        block = feature_meta[(feature_meta['variable'] == var) & (feature_meta['delay'] == 0)]
        picked.extend(block.head(n_station_each)['feature_index'].tolist())
    unique_picked = []
    for idx in picked:
        if idx not in unique_picked:
            unique_picked.append(idx)
    return unique_picked[:max_total]


def format_subset_description(subset_cfg):
    mode = subset_cfg.get('subset_mode', 'all')
    province_names = subset_cfg.get('province_names')
    city_names = subset_cfg.get('city_names')
    station_ids = subset_cfg.get('station_ids')
    return {
        'subset_mode': mode,
        'province_names': province_names,
        'city_names': city_names,
        'station_ids': station_ids,
    }


def build_map_panels_from_w(W, feature_meta, station_meta, variables=None):
    W_abs = np.abs(np.asarray(W, dtype=float).T)
    feature_meta = feature_meta.copy().reset_index(drop=True)
    station_meta = station_meta.copy().reset_index(drop=True)
    station_meta['station_id'] = station_meta['station_id'].astype(str)
    merged = feature_meta.join(pd.DataFrame(W_abs, columns=[f'z_{i+1}' for i in range(W_abs.shape[1])]))
    if variables is None:
        variables = list(pd.unique(merged['variable']))
    grouped_panels = {}
    for var in variables:
        sub = merged[merged['variable'] == var].copy()
        delays = sorted(pd.unique(sub['delay']))
        panels = []
        for delay in delays:
            delay_df = sub[sub['delay'] == delay].copy()
            delay_df['station_id'] = delay_df['station_id'].astype(str)
            panel_df = station_meta[['station_id', 'station_name', 'city', 'province', 'lon', 'lat']].merge(
                delay_df[['station_id'] + [f'z_{i+1}' for i in range(W_abs.shape[1])]],
                on='station_id',
                how='left',
            )
            panel_df = panel_df.sort_values('station_id').reset_index(drop=True)
            numeric_cols = [f'z_{i+1}' for i in range(W_abs.shape[1])]
            panel_df[numeric_cols] = panel_df[numeric_cols].fillna(0.0)
            panels.append({
                'data': panel_df[numeric_cols],
                'delay': int(delay),
                'title_suffix': f'_{var}',
            })
        grouped_panels[var] = panels
    return grouped_panels


def flatten_map_panels(grouped_panels, variable_order=None):
    if variable_order is None:
        variable_order = list(grouped_panels.keys())
    panels = []
    for var in variable_order:
        panels.extend(grouped_panels.get(var, []))
    return panels


def plot_prediction_comparison(predictions, targets, feature_names, title, n_show=4, length=160):
    n_show = min(n_show, targets.shape[1])
    fig, axes = plt.subplots(n_show, 1, figsize=(10, 2.6 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, range(n_show)):
        ax.plot(targets[:length, idx], label=f'真实 {feature_names[idx]}', linewidth=1.4)
        ax.plot(predictions[:length, idx], '--', label=f'预测 {feature_names[idx]}', linewidth=1.5)
        ax.legend(loc='upper right', fontsize=8)
        ax.set_ylabel('值')
    axes[0].set_title(title)
    axes[-1].set_xlabel('样本索引')
    plt.tight_layout()
    plt.show()


def compute_fft_summary(series, sampling_hours=1.0, top_k=3):
    series = np.asarray(series, dtype=float)
    centered = series - np.mean(series)
    spectrum = np.fft.rfft(centered)
    freqs = np.fft.rfftfreq(len(centered), d=sampling_hours)
    amplitudes = np.abs(spectrum)
    if len(amplitudes) > 0:
        amplitudes[0] = 0.0
    order = np.argsort(amplitudes)[::-1]
    rows = []
    for idx in order[:top_k]:
        freq = float(freqs[idx])
        period = np.inf if np.isclose(freq, 0.0) else 1.0 / freq
        rows.append({
            'frequency_per_hour': freq,
            'period_hours': period,
            'amplitude': float(amplitudes[idx]),
        })
    return pd.DataFrame(rows), freqs, amplitudes


## 第一部分：主流程

### 1.1 数据读取与绘制

先按 `exp_air_0321.ipynb` 的方式读取空气质量数据与站点元数据，再把所选变量整理为统一微观状态矩阵。这里先做三件事：

1. 检查数据读取是否正常；
2. 显示当前区域筛选方式和变量组合；
3. 先看少量微观曲线，建立对数据尺度和波动形态的直观印象。

因为站点较多，后面所有图都尽量只选少量代表性曲线，避免信息过载。


In [ ]:
dataset_path = locate_existing_path(config['dataset_path_candidates'], 'air dataset')
station_meta_path = locate_existing_path(config['station_meta_path_candidates'], 'station metadata')

air_subset = load_air_data_subset(
    dataset_path=dataset_path,
    station_meta_path=station_meta_path,
    subset_mode=config['subset']['subset_mode'],
    province_names=config['subset']['province_names'],
    city_names=config['subset']['city_names'],
    station_ids=config['subset']['station_ids'],
    variables=config['variables'],
    time_slice=config['subset']['time_slice'],
)

subset_summary = summarize_air_subset(air_subset)
air_matrix = build_air_feature_matrix(air_subset, variables=config['variables'])

x_data_raw = np.asarray(air_matrix['x_data_raw'], dtype=float)
t_data_raw = air_matrix['times']
station_meta_raw = air_matrix['station_meta'].reset_index(drop=True).copy()
feature_names_raw = [simplify_feature_name(name) for name in air_matrix['feature_names']]
feature_meta_raw = extract_feature_meta(feature_names_raw)
variables_normalized = [normalize_air_variable_name(v) for v in air_matrix['variables']]
sampling_hours = infer_sampling_hours(t_data_raw)
subset_desc = format_subset_description(config['subset'])

selected_micro_indices_raw = build_selected_micro_indices(
    feature_meta_raw,
    variables_normalized,
    n_station_each=config['selected_station_plot_count'],
    max_total=config['max_micro_plot_count'],
)

display(subset_summary['summary_df'])
display(subset_summary['province_counts'])
display(subset_summary['city_counts'].head(20))
print('Dataset path:', dataset_path)
print('Station meta path:', station_meta_path)
print('当前筛选配置:', subset_desc)
print('当前变量组合:', air_matrix['variables'])
print('Raw matrix shape:', x_data_raw.shape)
print('Sampling interval (hours):', sampling_hours)

plt.figure(figsize=(12, 4.5))
for idx in selected_micro_indices_raw:
    plt.plot(t_data_raw, x_data_raw[:, idx], linewidth=1.0, label=feature_names_raw[idx])
plt.title('原始微观空气质量曲线（少量代表站点）')
plt.xlabel('时间')
plt.ylabel('值')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

### 1.2 观测函数（时间延迟）、时间尺度与样本配对

按照研究框架，先设观测函数

$$
\mathbf{o}_t = g(\mathbf{x}_t),
$$

这里选择时间延迟嵌入作为 `g`，把当前值与若干过去时刻的空气质量一起拼成扩展观测。这样做的好处是：即使单时刻观测不够，也能借延迟坐标补充动力学记忆。

之后再按时间尺度 `tau` 构造样本配对：

$$
(\mathbf{o}_t, \mathbf{o}_{t+\tau}).
$$

这一步决定后续 `GIS` 拟合是在怎样的跨时间窗口上比较因果组织效率。


In [ ]:
H_list, delay_feature_names = lift_time_delay(
    x_data_raw,
    feature_names=feature_names_raw,
    n_delays=config['n_delays'],
    delay_interval=config['delay_interval'],
)
obs_air = np.asarray(H_list[0] if isinstance(H_list, list) else H_list, dtype=float)
obs_air = obs_air - obs_air.mean(axis=0, keepdims=True)
feature_names_air = [simplify_feature_name(name) for name in delay_feature_names]
feature_meta_air = extract_feature_meta(feature_names_air)
t_data_obs = t_data_raw[config['n_delays'] * config['delay_interval']:]

X_now_air, X_next_air = prepare_time_pairs(
    obs_air,
    tau=config['tau'],
    burn_in=config['burn_in'],
    stride=config['stride'],
)

selected_micro_indices = build_selected_micro_indices(
    feature_meta_air,
    variables_normalized,
    n_station_each=config['selected_station_plot_count'],
    max_total=config['max_micro_plot_count'],
)
selected_micro_names = [feature_names_air[idx] for idx in selected_micro_indices]

print('观测矩阵形状:', obs_air.shape)
print('配对样本形状:', X_now_air.shape, X_next_air.shape)
print('tau =', config['tau'], '?')
print('前 12 个观测特征名:', feature_names_air[:12])
print('选中的微观曲线:', selected_micro_names)

### 1.3 微观层 A/K_raw 与 Sigma 的拟合

在微观观测层上拟合线性 `GIS`：

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}_{\text{micro}}\mathbf{o}_t + \boldsymbol{\varepsilon}_t,
\qquad
\boldsymbol{\varepsilon}_t \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma}_{\text{micro}}).
$$

这里的 `A` 描述观测变量之间跨 `tau` 的有效传播结构，`Sigma` 则吸收未建模非线性、随机扰动和有限分辨率误差。由于微观维度很高，热力图标签会自动稀疏化显示。


In [ ]:
micro_fit = fit_linear_gis_from_pairs(
    X_now_air,
    X_next_air,
    fit_intercept=False,
    ridge=config['ridge'],
    regularization=config['eps'],
)
A_micro = micro_fit['A']
K_raw_micro = micro_fit['K_raw']
Sigma_micro = micro_fit['Sigma']

print('A_micro shape:', A_micro.shape)
print('K_raw_micro shape:', K_raw_micro.shape)
print('Sigma_micro shape:', Sigma_micro.shape)
print('||A_micro||_F =', np.linalg.norm(A_micro))
print('trace(Sigma_micro) =', np.trace(Sigma_micro))

plot_matrix_heatmap(
    A_micro,
    '微观层动力学矩阵 A',
    row_labels=feature_names_air,
    col_labels=feature_names_air,
    figsize=(7, 7),
    max_visible_labels=config['matrix_max_visible_labels'],
)
plot_matrix_heatmap(
    Sigma_micro,
    '微观层噪声协方差 Sigma',
    row_labels=feature_names_air,
    col_labels=feature_names_air,
    center=None,
    figsize=(7, 7),
    cmap='Blues',
    max_visible_labels=config['matrix_max_visible_labels'],
)

### 1.4 微观层预测与误差

得到微观层矩阵后，可以直接做单步和多步滚动预测。这里主要看两件事：

1. 预测曲线是否能跟住主要趋势；
2. 随着预测步数增加，误差是否按预期上升。

为了避免图像过密，只展示少量代表性的微观分量。


In [ ]:
micro_errors = compute_prediction_errors(
    A_micro,
    obs_air,
    tau=config['tau'],
    horizons=config['horizons'],
)

micro_error_df = pd.DataFrame({
    'horizon': list(micro_errors.keys()),
    'mean_error': [micro_errors[h]['mean_error'] for h in micro_errors.keys()],
})
display(micro_error_df)

micro_pred_1 = micro_errors[1]['predictions'][:, selected_micro_indices]
micro_target_1 = micro_errors[1]['targets'][:, selected_micro_indices]
plot_prediction_comparison(
    micro_pred_1,
    micro_target_1,
    selected_micro_names,
    title='微观层单步预测曲线对比（只展示少量分量）',
    n_show=len(selected_micro_names),
    length=160,
)

### 1.5 微观层 GIS 指标

接下来把微观层模型转成 `GIS` 指标。核心量包括：

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}^{\top}\boldsymbol{\Sigma}^{-1}\mathbf{A}),
$$

以及

$$
J_{\alpha} = \frac{1}{d}\left[\left(\frac12 - \frac{\alpha}{4}\right)N + \frac{\alpha}{4}D\right].
$$

这里 `D` 对应前向确定性，`N` 对应后向非简并性，`J_alpha` 则是做维度归一化后的综合效率，后面会拿它和宏观层做直接比较。


In [ ]:
micro_metrics = compute_gis_metrics(
    A_micro,
    Sigma_micro,
    alpha=config['alpha'],
    eps=config['eps'],
)

micro_metrics_df = pd.DataFrame({
    'metric': ['Gamma', 'log_Gamma', 'J_alpha', 'D', 'N'],
    'value': [
        micro_metrics['Gamma'],
        micro_metrics['log_Gamma'],
        micro_metrics['J_alpha'],
        micro_metrics['D'],
        micro_metrics['N'],
    ],
})
display(micro_metrics_df)

plot_dual_singular_spectra(
    micro_metrics['sv_forward'],
    micro_metrics['sv_backward'],
    title='微观层前向谱与后向谱对比',
    max_points=30,
)


### 1.6 宏观维度选择与 W 的构造

宏观层由粗粒化矩阵 `W` 给出：

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t,
$$

其中 `W` 的行向量代表保留下来的宏观方向。这里根据微观层谱结构选择宏观维度 `r`，并用 `SVD` 路线构造 `W`。若希望自由试验，也可以改成别的 `r` 或别的选秩模式。


In [ ]:
if config['rank_mode'] == 'manual':
    macro_r, rank_meta = select_macro_rank(
        micro_metrics['sv_backward'],
        mode='manual',
        manual_r=config['manual_r'],
        eps=config['eps'],
    )
else:
    macro_r, rank_meta = select_macro_rank(
        micro_metrics['sv_backward'],
        mode=config['rank_mode'],
        threshold=config['rank_threshold'],
        eps=config['eps'],
    )

w_result = build_w_from_svd(
    A_micro,
    Sigma_micro,
    r=macro_r,
    alpha=config['alpha'],
    eps=config['eps'],
    mode='two_stage',
)
W_air = w_result['W']
macro_names = [f'z_{i+1}' for i in range(macro_r)]

print('选定宏观维度 r =', macro_r)
print('rank_meta =', rank_meta)
print('W_air shape =', W_air.shape)

plot_matrix_heatmap(
    np.abs(W_air),
    '粗粒化矩阵 |W|',
    row_labels=macro_names,
    col_labels=feature_names_air,
    center=None,
    figsize=(10, 3.6),
    cmap='Blues',
    max_visible_labels=config['matrix_max_visible_labels'],
)

### 1.7 宏观数据生成

有了 `W` 之后，就能把高维微观观测投影成低维宏观变量：

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t.
$$

这一步的重点不是简单压缩，而是构造一个更适合描述跨时间传播结构的表示层。下面先看宏观时间序列本身，再和少量微观曲线做标准化后对比。


In [ ]:
z_air = apply_coarse_graining(W_air, obs_air)
Z_now_air, Z_next_air = prepare_time_pairs(
    z_air,
    tau=config['tau'],
    burn_in=config['burn_in'],
    stride=config['stride'],
)

print('宏观时间序列形状:', z_air.shape)
print('宏观配对样本形状:', Z_now_air.shape, Z_next_air.shape)

plt.figure(figsize=(12, 4.2))
for idx, name in enumerate(macro_names):
    plt.plot(t_data_obs, z_air[:, idx], linewidth=1.4, label=name)
plt.title('宏观变量时间序列')
plt.xlabel('时间')
plt.ylabel('宏观值')
plt.legend()
plt.tight_layout()
plt.show()

### 1.8 宏观层拟合

在宏观层上重复一次同样的线性 `GIS` 建模：

$$
\mathbf{z}_{t+\tau} \approx \mathbf{A}_{\text{macro}}\mathbf{z}_t + \boldsymbol{\varepsilon}^{(z)}_t,
\qquad
\boldsymbol{\varepsilon}^{(z)}_t \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma}_{\text{macro}}).
$$

由于宏观维度已经大幅下降，这里的矩阵结构通常会更紧凑，也更适合直接读图。


In [ ]:
macro_fit = fit_linear_gis_from_pairs(
    Z_now_air,
    Z_next_air,
    fit_intercept=False,
    ridge=config['ridge'],
    regularization=config['eps'],
)
A_macro = macro_fit['A']
Sigma_macro = macro_fit['Sigma']

print('A_macro shape:', A_macro.shape)
print('Sigma_macro shape:', Sigma_macro.shape)

plot_matrix_heatmap(
    A_macro,
    '宏观层动力学矩阵 A',
    row_labels=macro_names,
    col_labels=macro_names,
    figsize=(4.4, 4.0),
    max_visible_labels=macro_r,
)
plot_matrix_heatmap(
    Sigma_macro,
    '宏观层噪声协方差 Sigma',
    row_labels=macro_names,
    col_labels=macro_names,
    center=None,
    figsize=(4.4, 4.0),
    cmap='Blues',
    max_visible_labels=macro_r,
)

### 1.9 宏观层预测、误差

宏观层也要检查预测性能。这里关心的问题是：降维之后，模型是否仍保持良好的闭合性，以及误差是否相对微观层出现明显恶化。若宏观变量真抓住了主导结构，那么它在少量维度下也应保持较稳定的预测表现。


In [ ]:
macro_errors = compute_prediction_errors(
    A_macro,
    z_air,
    tau=config['tau'],
    horizons=config['horizons'],
)

macro_error_df = pd.DataFrame({
    'horizon': list(macro_errors.keys()),
    'mean_error': [macro_errors[h]['mean_error'] for h in macro_errors.keys()],
})
display(macro_error_df)

plot_prediction_comparison(
    macro_errors[1]['predictions'],
    macro_errors[1]['targets'],
    macro_names,
    title='宏观层单步预测曲线对比',
    n_show=len(macro_names),
    length=160,
)

### 1.10 宏观层GIS指标

宏观层同样计算 `Gamma`、`log Gamma`、`J_alpha`、`D`、`N`。只有这样，后面的 `CE` 才能真正回答“粗粒化之后单位维度的动力学效率有没有提高”，而不是只看误差曲线做经验判断。


In [ ]:
macro_metrics = compute_gis_metrics(
    A_macro,
    Sigma_macro,
    alpha=config['alpha'],
    eps=config['eps'],
)

macro_metrics_df = pd.DataFrame({
    'metric': ['Gamma', 'log_Gamma', 'J_alpha', 'D', 'N'],
    'value': [
        macro_metrics['Gamma'],
        macro_metrics['log_Gamma'],
        macro_metrics['J_alpha'],
        macro_metrics['D'],
        macro_metrics['N'],
    ],
})
display(macro_metrics_df)

plot_dual_singular_spectra(
    macro_metrics['sv_forward'],
    macro_metrics['sv_backward'],
    title='宏观层前向谱与后向谱对比',
)


### 1.11 CE值 + 宏微观曲线对比

现在正式比较宏微两层的维度平均效率。`CE` 的定义是：

$$
\mathrm{CE} = J_{\alpha}^{\text{macro}} - J_{\alpha}^{\text{micro}}.
$$

若 `CE > 0`，说明宏观表示虽然维度更低，但单位维度上的有效动力学组织更强。除了数值结果，也把宏微曲线放在一起做标准化可视化，帮助理解“压缩后的主导模式”到底长什么样。


In [ ]:
ce_air = compute_ce_from_micro_macro(micro_metrics, macro_metrics)
print('CE =', ce_air['CE'])
print('delta_D =', ce_air['delta_D'])
print('delta_N =', ce_air['delta_N'])

raw_aligned = x_data_raw[config['n_delays'] * config['delay_interval'] :]

plt.figure(figsize=(11.5, 4.8))
for idx in selected_micro_indices_raw:
    plt.plot(t_data_obs[:160], standardize_for_plot(raw_aligned[:160, idx]), linewidth=1.1, alpha=0.85, label=f'raw: {feature_names_raw[idx]}')
for idx, name in enumerate(macro_names):
    plt.plot(t_data_obs[:160], standardize_for_plot(z_air[:160, idx]), '--', linewidth=2.2, label=f'macro: {name}')
plt.title('原始数据与宏观曲线对比（标准化后展示）')
plt.xlabel('时间')
plt.ylabel('标准化值')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()


### 1.12 结果汇总

这一部分把参数、维度、误差和 `GIS` 指标集中成一条结果记录。这样做的目的有两个：

1. 方便后面做不同 `tau`、不同 `r`、不同区域筛选、不同变量组合的横向对比；
2. 为结尾摘要提供统一的数值依据。


In [ ]:
summary_dict, summary_row = summarize_pipeline_results(
    config={
        'experiment_name': config['experiment_name'],
        'tau': config['tau'],
        'delta': None,
        'alpha': config['alpha'],
        'rank_mode': config['rank_mode'],
        'manual_r': config['manual_r'],
        'subset': subset_desc,
        'variables': list(config['variables']),
    },
    micro_fit=micro_fit,
    macro_fit=macro_fit,
    micro_metrics=micro_metrics,
    macro_metrics=macro_metrics,
    prediction_results={'micro_errors': micro_errors, 'macro_errors': macro_errors},
    ce_result=ce_air,
    extra={
        'W': W_air,
        'rank_meta': rank_meta,
        'selected_micro_names': selected_micro_names,
        'sampling_hours': sampling_hours,
    },
)
summary_df = pd.DataFrame([summary_row])
display(summary_df)
summary_dict['core_numbers'] = {
    'micro_dim': micro_metrics['dimension'],
    'macro_dim': macro_metrics['dimension'],
    'micro_J_alpha': micro_metrics['J_alpha'],
    'macro_J_alpha': macro_metrics['J_alpha'],
    'CE': ce_air['CE'],
}
summary_dict['core_numbers']

## 第二部分：实验特有部分（主要参考 `exp_air_0321.ipynb`）

### 2.1 粗粒化矩阵的可视化归因，把相应因素放到地图上，方便看

这里重点分析粗粒化矩阵 `W` 的空间归因。思路和 `exp_air_0321.ipynb` 一致：把 `|W|` 中对应站点、变量、延迟层的系数投到地图上，看不同宏观模态主要由哪些区域和哪些时间延迟贡献。

由于空气实验的延迟层较多，这里将每个面板的尺寸控制在大约 `4 × 5`，并对不同面板共用颜色范围，便于横向比较。


In [ ]:
map_panels_grouped = build_map_panels_from_w(
    W_air,
    feature_meta_air,
    station_meta_raw,
    variables=variables_normalized,
)
map_panels = flatten_map_panels(map_panels_grouped, variable_order=variables_normalized)

all_map_values = []
for panel in map_panels:
    all_map_values.append(np.asarray(pd.DataFrame(panel['data']), dtype=float).ravel())
all_map_values = np.concatenate(all_map_values)
map_vmin = float(np.nanmin(all_map_values))
map_vmax = float(np.nanmax(all_map_values))

try:
    plot_station_with_map(
        station_meta_raw[['station_id', 'station_name', 'city', 'province', 'lon', 'lat']],
        map_panels,
        vmin=map_vmin,
        vmax=map_vmax,
        ncols=4,
        figsize=(14.5, 14.0),
        marker_size=28,
        edge_linewidth=0.28,
        colorbar_location='right',
        colorbar_label='|W| 权重',
    )
except ModuleNotFoundError as exc:
    print(f'跳过地图绘制，原因是缺少可选依赖: {exc}')


### 2.2 频谱分析解释块

除了时域曲线，还可以从频域看微观与宏观变量的主导周期。这里使用离散傅里叶变换，比较少量微观分量与宏观变量的主频位置和谱峰强度。

如果宏观变量真的抓住了主导组织结构，那么它的频谱通常会比单条微观曲线更集中，主峰也更容易解释。下面给出两个结果：

1. 微观代表分量与宏观变量的幅度谱曲线；
2. 各条曲线最强若干主频及对应周期。


In [ ]:
fft_summary_rows = []
plt.figure(figsize=(11, 4.8))

for idx in selected_micro_indices[: min(4, len(selected_micro_indices))]:
    fft_df, freqs, amplitudes = compute_fft_summary(obs_air[:, idx], sampling_hours=sampling_hours, top_k=3)
    fft_df['series'] = f'micro: {feature_names_air[idx]}'
    fft_summary_rows.append(fft_df)
    plt.plot(freqs[1:300], amplitudes[1:300], linewidth=1.2, alpha=0.8, label=f'micro: {feature_names_air[idx]}')

for idx, name in enumerate(macro_names):
    fft_df, freqs, amplitudes = compute_fft_summary(z_air[:, idx], sampling_hours=sampling_hours, top_k=3)
    fft_df['series'] = f'macro: {name}'
    fft_summary_rows.append(fft_df)
    plt.plot(freqs[1:300], amplitudes[1:300], '--', linewidth=2.0, label=f'macro: {name}')

plt.title('宏微观变量的频谱对比（截取低频部分）')
plt.xlabel('频率（每小时）')
plt.ylabel('值')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

fft_summary_df = pd.concat(fft_summary_rows, ignore_index=True)
display(fft_summary_df)

## 第三部分：结尾统一摘要

### 3.1 统一结论（主要用解释块对整个实验做汇总说明）

从本次空气质量 `GIS` 实验的主流程看，时间延迟嵌入把原始空气数据转成了更适合做跨时间建模的微观观测层；随后通过 `W` 构造出的宏观变量，在预测误差、谱结构和 `J_alpha` 上给出了比原始高维观测更紧凑的结果。

结论上主要看三点：

1. 微观层虽然保留了更完整的细节，但维度高、噪声协方差也更复杂；
2. 宏观层把主要传播结构压缩到少量模态后，单位维度效率通常更高；
3. 若 `CE` 为正，则说明这次粗粒化不是单纯丢信息，而是在当前观测和时间尺度下提高了动力学组织效率。


In [ ]:
final_summary = {
    'station_count': int(len(station_meta_raw)),
    'subset_mode': config['subset']['subset_mode'],
    'variables': ', '.join(config['variables']),
    'micro_dimension': int(micro_metrics['dimension']),
    'macro_dimension': int(macro_metrics['dimension']),
    'tau': int(config['tau']),
    'sampling_hours': float(sampling_hours),
    'micro_J_alpha': float(micro_metrics['J_alpha']),
    'macro_J_alpha': float(macro_metrics['J_alpha']),
    'CE': float(ce_air['CE']),
    'micro_error_h1': float(micro_errors[1]['mean_error']),
    'macro_error_h1': float(macro_errors[1]['mean_error']),
}

print('统一摘要：')
for key, value in final_summary.items():
    print(f'{key}: {value}')

display(pd.DataFrame([
    {'layer': 'micro', 'dimension': micro_metrics['dimension'], 'J_alpha': micro_metrics['J_alpha'], 'horizon_1_error': micro_errors[1]['mean_error']},
    {'layer': 'macro', 'dimension': macro_metrics['dimension'], 'J_alpha': macro_metrics['J_alpha'], 'horizon_1_error': macro_errors[1]['mean_error']},
]))